# Tiny ImageNet provenance, separation, and paired input streams

Purpose: importable real-data foundation for Notebook 01. Inputs: public, pinned Parquet shards from `thethinkmachine/tiny-imagenet`. Outputs: verified cache and exact source/split/subset manifests below `data/tiny_imagenet/`. First complete download and decoded-RGB audit takes approximately 1–5 minutes; subsequent loads normally take seconds. This module does not train or evaluate a model and has no checkpoint state. Resume interrupted download/audit by calling `prepare_dataset(PROJECT_ROOT, config)` again: completed, hashed files are reused, partial files are rewritten, and manifests are installed atomically after success. No remote dataset code is executed.

Definitions alone carry `ahr-export`; consuming notebooks use the shared importer, never execute experiment cells. All randomness below uses local generators keyed by sample ID, occurrence and distinct stream seed.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from functools import lru_cache
from datetime import datetime, timezone
from urllib.parse import quote
import hashlib, io, json, math, os, shutil, urllib.request, uuid
import numpy as np
import torch
from PIL import Image
import pyarrow.parquet as pq
from fmm_jepa.data import sample_masks

DATA_SCHEMA = "fmm-jepa-tiny-data/v1"
SOURCE_REPOSITORY = "thethinkmachine/tiny-imagenet"
SOURCE_REVISION = "9f4331d7160a7f6e90c12e3533d658e9c8c7ae3c"
PARQUET_REVISION = "5318f3a410192b1480adce2ea61e8e2c139adb53"
AUDIT_VERSION = "decoded-rgb-sha256/v1"
DUPLICATE_POLICY = "candidate stratified 450/50; decoded-RGB groups prioritize heldout>development>training; keep lowest original row in winning partition; exclude all others before subset selection"

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.name + f".{os.getpid()}.{uuid.uuid4().hex}.partial")
    with partial.open("w") as stream:
        json.dump(value, stream, indent=2, sort_keys=True, allow_nan=False)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(partial, path)

def _get_json(url):
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent":"fmm-jepa-research-notebook/0.1"}), timeout=90) as response:
        return json.load(response)

def sample_id(split, row):
    return f"{SOURCE_REPOSITORY}@{PARQUET_REVISION}:{split}:{int(row)}"

def parse_sample_id(identifier):
    prefix, split, row = identifier.rsplit(":", 2)
    if prefix != f"{SOURCE_REPOSITORY}@{PARQUET_REVISION}" or split not in ("train", "valid"):
        raise ValueError("sample identity belongs to a different source/revision/split")
    return split, int(row)

def stable_seed(base_seed, stream, *identity):
    # 63 bits is accepted by torch generators and does not depend on Python hash randomization.
    return int(canonical_hash([int(base_seed), stream, *identity])[:16], 16) % (2**63 - 1)

def _data_config(config):
    return config.get("data", {})

def _stream_seed(config, stream, default):
    seeds = config.get("seeds", {})
    aliases = {"augmentation": ("augmentation", "augment"), "mask": ("mask", "masking")}
    return int(next((seeds[k] for k in aliases.get(stream, (stream,)) if k in seeds), default))


## Immutable source verification and bounded cache

The card is not executable authority: its copied loading example points at another repository. We discover actual split names through the Viewer API, cross-check pinned source metadata, and download only supported Parquet files from the intended repository at the immutable converted revision. Cached file hashes are checked every time. The complete compressed source is about 161 MB; selection occurs after metadata and duplicate auditing, so a smoke subset does not imply that only selected examples were downloaded.

The repository API reports public/ungated access. The source card has an empty license field and includes ImageNet terms describing non-commercial research/educational use. Those declarations are recorded without assigning a license or claiming broader permission.

In [ ]:
def ensure_source(project_root):
    cache = Path(project_root) / "data" / "tiny_imagenet" / PARQUET_REVISION
    cache.mkdir(parents=True, exist_ok=True)
    manifest_path = cache / "dataset_manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("schema") != DATA_SCHEMA or manifest.get("parquet_revision") != PARQUET_REVISION:
            raise ValueError("incompatible cached dataset manifest; inspect instead of silently replacing it")
        if manifest.get("source_revision") != SOURCE_REVISION or manifest.get("repository") != SOURCE_REPOSITORY:
            raise ValueError("cached source provenance mismatch")
        if canonical_hash(manifest["class_names"]) != manifest.get("class_mapping_sha256"):
            raise ValueError("cached class mapping hash mismatch")
        expected_fingerprint=canonical_hash({"repo":SOURCE_REPOSITORY,"revision":PARQUET_REVISION,
            "file_hashes":[f["sha256"] for f in manifest["files"]],"classes":manifest["class_names"]})
        if expected_fingerprint != manifest.get("dataset_fingerprint"):
            raise ValueError("cached dataset fingerprint mismatch")
        for info in manifest["files"]:
            path = cache / info["cache_filename"]
            if not path.exists() or path.stat().st_size != info["bytes"] or file_sha256(path) != info["sha256"]:
                raise ValueError(f"missing/corrupt immutable dataset cache: {path}")
        return cache, manifest
    base = f"https://huggingface.co/api/datasets/{SOURCE_REPOSITORY}"
    source = _get_json(base + "/revision/" + SOURCE_REVISION)
    converted = _get_json(base + "/revision/" + PARQUET_REVISION)
    if source["id"] != SOURCE_REPOSITORY or source["sha"] != SOURCE_REVISION or converted["sha"] != PARQUET_REVISION:
        raise ValueError("pinned source identity mismatch")
    viewer_splits = _get_json("https://datasets-server.huggingface.co/splits?dataset=" + SOURCE_REPOSITORY)
    discovered = sorted(x["split"] for x in viewer_splits.get("splits", []) if x["config"] == "default")
    if discovered != ["train", "valid"] or viewer_splits.get("failed"):
        raise ValueError(f"source split contract changed: {discovered}")
    paths = sorted(x["rfilename"] for x in converted["siblings"] if x["rfilename"].endswith(".parquet"))
    if sorted(p.split("/")[1] for p in paths) != discovered:
        raise ValueError("converted shard paths do not match discovered source splits")
    if shutil.disk_usage(cache).free < 1024**3:
        raise RuntimeError("need at least 1 GiB free for complete compressed data, audit, and manifests")
    info_url = f"https://huggingface.co/datasets/{SOURCE_REPOSITORY}/raw/{SOURCE_REVISION}/dataset_infos.json"
    pinned_infos = _get_json(info_url)
    if len(pinned_infos) != 1:
        raise ValueError("expected a single default dataset info record")
    info_key, pinned_info = next(iter(pinned_infos.items()))
    pinned_features = pinned_info["features"]
    source_tree = _get_json(base + "/tree/" + SOURCE_REVISION + "/data?expand=true")
    source_files = {item["path"].split("/")[-1].split("-")[0]:item for item in source_tree if item["path"].endswith(".parquet")}
    files, mappings, split_metadata = [], {}, {}
    for remote_path in paths:
        split = remote_path.split("/")[1]
        url = f"https://huggingface.co/datasets/{SOURCE_REPOSITORY}/resolve/{PARQUET_REVISION}/{remote_path}"
        filename = split + "-0000.parquet"; path = cache / filename
        if not path.exists():
            temporary = path.with_suffix(".parquet.partial")
            with urllib.request.urlopen(url, timeout=120) as response, temporary.open("wb") as out:
                shutil.copyfileobj(response, out, length=8*1024*1024)
                out.flush(); os.fsync(out.fileno())
            os.replace(temporary, path)
        parquet = pq.ParquetFile(path)
        schema = parquet.schema_arrow
        if schema.names != ["image", "label"]:
            raise ValueError(f"unexpected columns: {schema.names}")
        # Conversion stripped HF schema metadata. Bind typed Arrow columns to the
        # pinned source dataset_infos mapping and cross-check both live split schemas.
        hf = pinned_features
        preview = _get_json("https://datasets-server.huggingface.co/first-rows?dataset="+SOURCE_REPOSITORY+"&config=default&split="+split)
        viewer_features = {item["name"]:item["type"] for item in preview["features"]}
        mapping = viewer_features["label"]["names"]
        if mapping != hf["label"]["names"] or viewer_features["image"]["_type"] != "Image":
            raise ValueError("live split schema differs from pinned source metadata")
        verified_file_hash = file_sha256(path)
        if verified_file_hash != source_files[split]["lfs"]["oid"] or path.stat().st_size != source_files[split]["size"]:
            raise ValueError("converted Parquet differs from pinned original source file")
        if len(mapping) != 200 or len(set(mapping)) != 200 or hf["image"]["_type"] != "Image":
            raise ValueError("unexpected image/class schema")
        labels = parquet.read(columns=["label"]).column("label").to_numpy()
        counts = Counter(map(int, labels))
        expected_rows, expected_count = (100000, 500) if split == "train" else (10000, 50)
        if parquet.metadata.num_rows != expected_rows or counts != Counter({i: expected_count for i in range(200)}):
            raise ValueError(f"unexpected {split} counts/labels; cannot apply registered split unchanged")
        mappings[split] = mapping
        split_metadata[split] = {"rows":len(labels), "class_counts":{str(k):v for k,v in sorted(counts.items())},
                                 "schema":str(schema), "image_feature":hf["image"], "label_feature":hf["label"]}
        files.append({"split":split,"remote_path":remote_path,"url":url,"cache_filename":filename,
                      "bytes":path.stat().st_size,"sha256":verified_file_hash,"source_lfs_sha256":source_files[split]["lfs"]["oid"],"rows":len(labels)})
    if mappings["train"] != mappings["valid"]:
        raise ValueError("class-name mapping differs across source splits")
    card = source.get("cardData", {})
    manifest = {"schema":DATA_SCHEMA,"repository":SOURCE_REPOSITORY,"source_revision":SOURCE_REVISION,
                "parquet_revision":PARQUET_REVISION,"verified_at_utc":datetime.now(timezone.utc).isoformat(),
                "verification_sources":{"pinned_source_api":base+"/revision/"+SOURCE_REVISION,
                    "pinned_parquet_api":base+"/revision/"+PARQUET_REVISION,
                    "viewer_splits":viewer_splits},
                "declared_access":{"private":source.get("private"),"gated":source.get("gated"),
                    "license":card.get("license"),"access_terms":card.get("extra_gated_prompt"),
                    "interpretation":"public/ungated API does not establish unrestricted reuse rights; no license inferred"},
                "files":files,"source_splits":split_metadata,"class_names":mappings["train"],
                "class_mapping_sha256":canonical_hash(mappings["train"]),
                "class_mapping_evidence":{"pinned_dataset_infos_url":info_url,"legacy_metadata_key":info_key,"both_viewer_split_mappings_equal":True,"converted_shards_byte_identical_to_source_lfs":True,"parquet_hf_metadata":"absent; pinned dataset_infos and both viewer split schemas cross-checked"},
                "downloaded_source_bytes":sum(f["bytes"] for f in files),
                "load_policy":"pinned Parquet only; no remote code; lazy image decoding",
                "heldout_source_split":"valid","heldout_policy":"reserved; decoding only for integrity/duplicate audit, no model evaluation by default"}
    manifest["dataset_fingerprint"] = canonical_hash({"repo":SOURCE_REPOSITORY,"revision":PARQUET_REVISION,
                    "file_hashes":[f["sha256"] for f in files],"classes":mappings["train"]})
    atomic_json(cache / "pinned_dataset_infos.json", pinned_infos)
    atomic_json(cache / "pinned_source_file_metadata.json", source_tree)
    atomic_json(cache / "source_repository_metadata.json", source)
    atomic_json(cache / "converted_repository_metadata.json", converted)
    atomic_json(manifest_path, manifest)
    return cache, manifest

class TinyImageNetAdapter:
    """Keep compressed image bytes in Arrow; decode one selected sample on demand."""
    def __init__(self, cache, manifest):
        self.cache = Path(cache); self.manifest = manifest
        self.tables = {f["split"]:pq.read_table(self.cache / f["cache_filename"], memory_map=True) for f in manifest["files"]}
        self.labels = {s:t.column("label").to_numpy() for s,t in self.tables.items()}
        self.class_names = manifest["class_names"]
        self.fingerprint = manifest["dataset_fingerprint"]
    def get(self, identifier):
        split, row = parse_sample_id(identifier)
        if row < 0 or row >= len(self.labels[split]):
            raise IndexError(identifier)
        item = self.tables[split].column("image")[row].as_py()
        if item["bytes"] is None:
            raise ValueError(f"external image path without embedded bytes: {identifier}")
        try:
            with Image.open(io.BytesIO(item["bytes"])) as original:
                dimensions, mode = list(original.size), original.mode
                image = original.convert("RGB"); image.load()
        except Exception as exc:
            raise RuntimeError(f"decode failed for {identifier}; no silent exclusions") from exc
        return {"image":image,"label":int(self.labels[split][row]),"sample_id":identifier,
                "source_dimensions":dimensions,"source_mode":mode,"split":split}
    def __len__(self):
        return sum(len(labels) for labels in self.labels.values())


def audit_source(dataset):
    """Decode all source images once for identity/integrity; training remains lazy."""
    path = dataset.cache / "decoded_rgb_audit.json"
    if path.exists():
        audit = json.loads(path.read_text())
        if audit.get("dataset_fingerprint") != dataset.fingerprint or audit.get("version") != AUDIT_VERSION:
            raise ValueError("stale decoded-image audit; inspect before replacing")
        if canonical_hash({k:v for k,v in audit.items() if k != "audit_sha256"}) != audit.get("audit_sha256"):
            raise ValueError("decoded-image audit hash mismatch")
        return audit
    seen = {}; duplicate_groups = {}; failures = []; modes = Counter(); dimensions = Counter()
    image_digest = hashlib.sha256()
    for split in ("train", "valid"):
        for row in range(len(dataset.labels[split])):
            identifier = sample_id(split, row)
            try:
                item = dataset.get(identifier)
                image = item["image"]
                modes[f"{split}:{item['source_mode']}"] += 1
                dimensions[f"{split}:{image.width}x{image.height}"] += 1
                digest = hashlib.sha256(f"RGB:{image.width}:{image.height}:".encode()+image.tobytes()).hexdigest()
                image_digest.update(f"{identifier}:{digest}\n".encode())
                if digest in seen:
                    duplicate_groups.setdefault(digest, [seen[digest]]).append(identifier)
                else:
                    seen[digest] = identifier
            except Exception as exc:
                failures.append({"sample_id":identifier,"error":repr(exc)})
    audit = {"version":AUDIT_VERSION,"dataset_fingerprint":dataset.fingerprint,"images_attempted":len(dataset),
             "images_decoded":len(dataset)-len(failures),"source_modes":dict(modes),"source_dimensions":dict(dimensions),
             "failures":failures,"duplicate_groups":duplicate_groups,"duplicate_group_count":len(duplicate_groups),
             "duplicate_extra_rows":sum(len(x)-1 for x in duplicate_groups.values()),
             "decoded_sequence_sha256":image_digest.hexdigest(),"scope":"all 110000 source images, exact decoded RGB dimensions+pixels",
             "limitations":"near duplicates, crops, semantic duplicates and approximate matches are not detected",
             "declared_exclusion_rule":DUPLICATE_POLICY}
    audit["audit_sha256"] = canonical_hash(audit)
    atomic_json(path, audit)
    return audit


## Separation before subset selection

Labels are used solely to construct reproducible balanced candidate pools and labeled frozen-feature evaluation. They never enter the JEPA forward call. Within each class a local split generator permutes 500 source-training rows, reserving 50 for development and 450 for the pretraining/probe pool. All source `valid` rows are held out.

The duplicate rule is registered before examining duplicate results: for identical decoded RGB images, reserve the held-out partition first, then development, then training; retain the lowest original row in the winning partition. Exclude all other members before choosing smoke/pilot subsets. This can reduce the 450/50 candidate pool sizes and is reported. Decoding held-out images for data integrity is not predictive evaluation. Corrupt images cause a visible error after recording the audit; there is no silent filtering. Neither preprocessing statistics nor model selection use held-out images.

In [ ]:
def construct_splits(dataset, audit, config):
    if audit["failures"]:
        raise RuntimeError("source decode failures recorded in decoded_rgb_audit.json; explicit protocol decision required")
    if set(audit["source_dimensions"]) != {"train:64x64", "valid:64x64"}:
        raise ValueError("native image-size contract differs from 64x64")
    data_config = _data_config(config)
    split_seed = int(data_config.get("split_seed", config.get("seeds", {}).get("split", 1701)))
    subset_seed = int(data_config.get("subset_seed", config.get("seeds", {}).get("subset", 1702)))
    profile = config.get("run_mode", config.get("profile", config.get("run", {}).get("profile", "smoke")))
    defaults = {"smoke":(10,5), "pilot":(100,10)}
    if profile not in defaults:
        raise ValueError("data profile must be smoke or pilot")
    per_train = data_config.get("train_per_class", defaults[profile][0])
    per_dev = data_config.get("dev_per_class", defaults[profile][1])
    per_train = "all" if per_train == "all" else int(per_train)
    per_dev = "all" if per_dev == "all" else int(per_dev)
    if any(value != "all" and value < 1 for value in (per_train, per_dev)):
        raise ValueError("all classes need positive subset counts")
    partitions = {"training":[],"development":[],"heldout":[]}
    membership = {}
    for label in range(200):
        rows = np.flatnonzero(dataset.labels["train"] == label)
        generator = np.random.default_rng(stable_seed(split_seed,"split",label))
        shuffled = generator.permutation(rows)
        for part, selected in (("development",shuffled[:50]),("training",shuffled[50:])):
            for row in selected:
                identifier = sample_id("train", int(row)); partitions[part].append(identifier); membership[identifier] = part
    for row in range(len(dataset.labels["valid"])):
        identifier = sample_id("valid", row); partitions["heldout"].append(identifier); membership[identifier] = "heldout"
    initial = {part:len(ids) for part,ids in partitions.items()}
    exclusions = []; priority = {"heldout":0,"development":1,"training":2}
    for digest, group in sorted(audit["duplicate_groups"].items()):
        ordered = sorted(group, key=lambda identifier:(priority[membership[identifier]],parse_sample_id(identifier)[1]))
        retained = ordered[0]
        for identifier in ordered[1:]:
            exclusions.append({"sample_id":identifier,"partition":membership[identifier],"reason":"exact_decoded_RGB_duplicate",
                               "retained_sample_id":retained,"retained_partition":membership[retained],"rgb_sha256":digest})
    excluded = {record["sample_id"] for record in exclusions}
    pools = {part:sorted(identifier for identifier in ids if identifier not in excluded) for part,ids in partitions.items()}
    def stratified_subset(part, count):
        by_class = defaultdict(list)
        for identifier in pools[part]:
            split,row = parse_sample_id(identifier); by_class[int(dataset.labels[split][row])].append(identifier)
        selected = []
        for label in range(200):
            if count != "all" and len(by_class[label]) < count:
                raise ValueError(f"insufficient {part} class {label} after duplicate policy: {len(by_class[label])} < {count}")
            generator = np.random.default_rng(stable_seed(subset_seed,"subset",part,label))
            selected.extend(np.asarray(by_class[label],dtype=object)[generator.permutation(len(by_class[label]))[:None if count == "all" else count]].tolist())
        return selected
    selected = {"training":stratified_subset("training",per_train),
                "development":stratified_subset("development",per_dev),"heldout":pools["heldout"]}
    assert not(set(pools["training"])&set(pools["development"]) or set(pools["training"])&set(pools["heldout"]) or set(pools["development"])&set(pools["heldout"]))
    manifest = {"schema":DATA_SCHEMA,"dataset_fingerprint":dataset.fingerprint,"parquet_revision":PARQUET_REVISION,
        "split_seed":split_seed,"subset_seed":subset_seed,"profile":profile,"candidate_pool_sizes":initial,
        "duplicate_policy":DUPLICATE_POLICY,"duplicate_audit_sha256":audit["audit_sha256"],"exclusions":exclusions,
        "pool_sizes":{p:len(ids) for p,ids in pools.items()},"pool_ids":pools,
        "pool_class_counts":{p:dict(sorted(Counter(str(int(dataset.labels[parse_sample_id(i)[0]][parse_sample_id(i)[1]])) for i in ids).items(), key=lambda x:int(x[0]))) for p,ids in pools.items()},
        "selected_ids":selected,"selected_sizes":{p:len(ids) for p,ids in selected.items()},
        "selected_per_class":{"training":per_train,"development":per_dev},
        "sample_id_disjoint":True,"exact_RGB_cross_partition_duplicates_remaining":0,
        "stratification_uses_labels":True,"probe_training_reuses_pretraining_pool":True,
        "heldout_predictive_evaluation":"NOT_RUN; reserved", "pretraining_labels_policy":"labels omitted from loss arguments"}
    manifest["split_sha256"] = canonical_hash(manifest)
    split_path = dataset.cache / f"split_{profile}_{manifest['split_sha256'][:16]}.json"
    atomic_json(split_path, manifest)
    return manifest


def prepare_dataset(project_root, config):
    for name, expected in (("repo_id", SOURCE_REPOSITORY), ("source_revision", SOURCE_REVISION), ("parquet_revision", PARQUET_REVISION)):
        if _data_config(config).get(name, expected) != expected:
            raise ValueError(f"configured dataset {name} does not match verified source")
    cache, manifest = ensure_source(project_root)
    dataset = TinyImageNetAdapter(cache, manifest)
    audit = audit_source(dataset)
    split = construct_splits(dataset, audit, config)
    enriched = dict(manifest)
    enriched["integrity_audit"] = {k:v for k,v in audit.items() if k not in ("duplicate_groups",)}
    conflicts=[]
    for digest, group in audit["duplicate_groups"].items():
        labels=[int(dataset.labels[parse_sample_id(identifier)[0]][parse_sample_id(identifier)[1]]) for identifier in group]
        if len(set(labels))>1:
            conflicts.append({"rgb_sha256":digest,"sample_ids":group,"source_labels":labels})
    enriched["integrity_audit"]["duplicate_label_conflict_count"]=len(conflicts)
    enriched["integrity_audit"]["duplicate_label_conflicts"]=conflicts
    enriched["integrity_audit"]["label_policy"]="preserve source label of deterministic retained member; no relabeling; source ambiguities remain a limitation"
    enriched["integrity_audit"]["exclusion_summary"] = dict(Counter(record["partition"]+"->"+record["retained_partition"] for record in split["exclusions"]))
    enriched["integrity_audit_path"] = str(cache / "decoded_rgb_audit.json")
    return {"dataset":dataset,"dataset_manifest":enriched,"split_manifest":split,"audit":audit,
            "train_ids":split["selected_ids"]["training"],"dev_ids":split["selected_ids"]["development"],
            "heldout_ids":split["selected_ids"]["heldout"]}


## Paired augmentation and multiblock masks

Every image/occurrence receives independent augmentation and mask seeds from a SHA-256-derived seed stream; model execution, global RNG consumption, data batching and evaluation cannot change later inputs. Training uses one PIL bilinear random resized crop (area 0.6–1.0, aspect 0.75–4/3) and a 0.5 horizontal flip, shared by teacher and student. Evaluation uses the native full 64×64 RGB image with deterministic resizing only if configured. Fixed mean/std `(0.5,0.5,0.5)` is a declared convention, not a statistic estimated from any split.

The existing verified multiblock sampler proposes four target rectangles. We retry the complete mask at most 16 times until there are at least four context tokens and all proposed target blocks have more than one token. Target-block overlap is intentional; context never overlaps their union. The final deterministic fallback repeats a 3×3 target rectangle (9/64 area) four times and uses its complement as context; its use and realized integer geometry are recorded. This fallback is a declared departure from the proposal range, not a silent relaxation of isolation.

In [ ]:
def preprocessing_descriptor(config=None):
    config = config or {}
    policy = config.get("preprocessing", {})
    mean = policy.get("mean", [0.5,0.5,0.5]); std = policy.get("std", [0.5,0.5,0.5])
    if len(mean) != 3 or len(std) != 3 or min(std) <= 0:
        raise ValueError("normalization requires three channel means and positive standard deviations")
    return {"version":"tiny-native-rgb/v1","resolution":int(config.get("image_size",64)),
            "training":{"crop_area":list(policy.get("crop_scale",[0.6,1.0])),"crop_aspect":list(policy.get("crop_ratio",[0.75,4/3])),"crop_attempts":10,
                        "interpolation":"PIL bilinear","horizontal_flip_probability":float(policy.get("horizontal_flip",0.5))},
            "evaluation":"complete native RGB image; bilinear resize only if dimensions differ",
            "normalization":{"mean":list(mean),"std":list(std),"learned_from_data":False}}


def preprocess_image(image, config=None, seed=0, training=False):
    descriptor = preprocessing_descriptor(config)
    resolution = descriptor["resolution"]
    if resolution != 64:
        raise ValueError("Notebook01 primary protocol is native 64x64; register another protocol to change it")
    image = image.convert("RGB")
    generator = np.random.default_rng(int(seed))
    width,height = image.size
    crop = [0,0,width,height]; flipped=False; fallback=False
    if training:
        for attempt in range(10):
            area = width * height * generator.uniform(*descriptor["training"]["crop_area"])
            aspect = math.exp(generator.uniform(*map(math.log,descriptor["training"]["crop_aspect"])))
            w,h = round(math.sqrt(area*aspect)),round(math.sqrt(area/aspect))
            if 0 < w <= width and 0 < h <= height:
                left=int(generator.integers(0,width-w+1));top=int(generator.integers(0,height-h+1))
                crop=[left,top,left+w,top+h];break
        else:
            side=min(width,height); left=(width-side)//2;top=(height-side)//2
            crop=[left,top,left+side,top+side];fallback=True
        image=image.crop(tuple(crop)).resize((resolution,resolution),Image.Resampling.BILINEAR)
        flipped=bool(generator.random()<descriptor["training"]["horizontal_flip_probability"])
        if flipped:image=image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    elif image.size != (resolution,resolution):
        image=image.resize((resolution,resolution),Image.Resampling.BILINEAR)
    values=np.asarray(image,dtype=np.uint8).copy()
    tensor=torch.from_numpy(values).permute(2,0,1).float().div_(255)
    mean=torch.tensor(descriptor["normalization"]["mean"])[:,None,None]
    std=torch.tensor(descriptor["normalization"]["std"])[:,None,None]
    tensor=(tensor-mean)/std
    metadata={"seed":int(seed),"training":bool(training),"crop_ltrb":crop,"flipped":flipped,
              "crop_fallback":fallback,"preprocessing_sha256":canonical_hash(descriptor)}
    return tensor,metadata


def _mask_one(grid, seed, minimum_context=4, attempts=16, policy=None):
    policy=policy or {}
    minimum_context=int(policy.get("min_context",minimum_context))
    if int(policy.get("target_count",4)) != 4 or not 1 <= minimum_context <= 55 or attempts < 1:
        raise ValueError("registered masks require four targets, min_context in [1,55], and bounded positive attempts")
    if grid != (8,8):
        raise ValueError("registered masks currently require primary 8x8 patch grid")
    generator=torch.Generator().manual_seed(int(seed))
    chosen=None
    for retry in range(attempts):
        proposal=sample_masks(grid,1,generator,context_area=policy.get("context_area",(0.85,1.0)),target_area=policy.get("target_area",(0.15,0.20)),
                              aspect_ratio=policy.get("aspect_ratio",(0.75,1.5)),target_count=4,max_retries=int(policy.get("max_retries",32)))
        if int(proposal["context_valid"].sum())>=minimum_context and all(int(v.sum())>1 for v in proposal["target_valid"]):
            chosen=proposal;break
    fallback=chosen is None
    if fallback:
        target=torch.tensor([[r*8+c for r in range(3) for c in range(3)]],dtype=torch.long)
        context=torch.tensor([[i for i in range(64) if i not in set(target[0].tolist())]],dtype=torch.long)
        chosen={"context_indices":context,"context_valid":torch.ones_like(context,dtype=torch.bool),
                "target_indices":[target.clone() for _ in range(4)],
                "target_valid":[torch.ones_like(target,dtype=torch.bool) for _ in range(4)],"metadata":{}}
    context=chosen["context_indices"][0][chosen["context_valid"][0]]
    blocks=[indices[0][valid[0]] for indices,valid in zip(chosen["target_indices"],chosen["target_valid"])]
    if len(context)<minimum_context or any(bool(torch.isin(context,t).any()) for t in blocks):
        raise AssertionError("minimum-context or target isolation contract violated")
    block_details=[]
    for block in blocks:
        rows=block//grid[1];cols=block%grid[1]
        block_details.append({"count":len(block),"height":int(rows.max()-rows.min()+1),
                              "width":int(cols.max()-cols.min()+1),"area_fraction":len(block)/64})
    identity=canonical_hash({"context":context.tolist(),"targets":[b.tolist() for b in blocks]})
    chosen["metadata"]={"seed":int(seed),"identity":identity,"minimum_context":minimum_context,
                        "context_count":len(context),"target_blocks":block_details,
                        "whole_mask_retry":retry,"max_whole_mask_attempts":attempts,"fallback":fallback,
                        "target_overlap_allowed":True,"grid":list(grid)}
    return chosen


def _pad_rows(rows):
    width=max(len(row) for row in rows)
    indices=torch.full((len(rows),width),-1,dtype=torch.long)
    valid=torch.zeros_like(indices,dtype=torch.bool)
    for index,row in enumerate(rows):
        indices[index,:len(row)]=row;valid[index,:len(row)]=True
    return indices,valid


def make_batch(dataset, sample_ids, occurrences, config, training=True):
    if len(sample_ids)!=len(occurrences) or not sample_ids:
        raise ValueError("nonempty paired sample IDs and occurrence numbers required")
    patch_size=int(config.get("model",{}).get("patch_size",config.get("patch_size",8)))
    if patch_size != 8:
        raise ValueError("patch-size4 is a separately registered future config; not the current data profile")
    images=[];labels=[];augments=[];mask_records=[];masks=[]
    for identifier,occurrence in zip(sample_ids,occurrences):
        if int(occurrence)<0:raise ValueError("occurrence must be nonnegative")
        item=dataset.get(identifier)
        seed=stable_seed(_stream_seed(config,"augmentation",1704),"augmentation",identifier,int(occurrence)) if training else 0
        image,augmentation=preprocess_image(item["image"],config,seed,training)
        augmentation.update({"sample_id":identifier,"occurrence":int(occurrence)})
        images.append(image);labels.append(item["label"]);augments.append(augmentation)
        mask_seed=stable_seed(_stream_seed(config,"mask",1705),"mask",identifier,int(occurrence))
        mask=_mask_one((8,8),mask_seed,policy=config.get("masking",{})); masks.append(mask);mask_records.append(mask["metadata"])
    ci,cv=_pad_rows([m["context_indices"][0][m["context_valid"][0]] for m in masks])
    targets=[_pad_rows([m["target_indices"][k][0][m["target_valid"][k][0]] for m in masks]) for k in range(4)]
    packed={"context_indices":ci,"context_valid":cv,"target_indices":[t[0] for t in targets],
            "target_valid":[t[1] for t in targets],"metadata":{"examples":mask_records,"grid":[8,8],"target_overlap_allowed":True}}
    return {"images":torch.stack(images),"labels":torch.tensor(labels,dtype=torch.long),"sample_ids":list(sample_ids),
            "occurrences":list(map(int,occurrences)),"augmentation_ids":[canonical_hash(a) for a in augments],
            "mask_ids":[m["identity"] for m in mask_records],"augmentations":augments,"masks":packed,
            "mask_metadata":mask_records,"preprocessing":preprocessing_descriptor(config)}


def evaluation_batch(dataset, sample_ids, config=None):
    # No mask creation or training RNG access is needed for frozen full-image features.
    samples=[dataset.get(identifier) for identifier in sample_ids]
    return {"images":torch.stack([preprocess_image(item["image"],config,training=False)[0] for item in samples]),
            "labels":torch.tensor([item["label"] for item in samples],dtype=torch.long),"sample_ids":list(sample_ids)}


In [ ]:
class EvaluationDataset(torch.utils.data.Dataset):
    """Deterministic full-image adapter; does not draw training or model randomness."""
    def __init__(self, dataset, sample_ids, config=None):
        self.dataset=dataset; self.sample_ids=list(sample_ids); self.config=config or {}
        self.fingerprint=canonical_hash({"dataset":dataset.fingerprint,"sample_ids":self.sample_ids,
                                        "preprocessing":preprocessing_descriptor(self.config)})
    def __len__(self):
        return len(self.sample_ids)
    def __getitem__(self,index):
        item=self.dataset.get(self.sample_ids[index])
        return {"image":preprocess_image(item["image"],self.config,training=False)[0],
                "label":item["label"],"sample_id":item["sample_id"]}


In [ ]:
print({"schema": DATA_SCHEMA, "repository": SOURCE_REPOSITORY, "source_revision": SOURCE_REVISION, "parquet_revision": PARQUET_REVISION, "network_access_on_import": False})